# Planning section
Includes fields to keep/ignore for each data source (NUS, NTU, SUTD courses, and job descriptions), and the target schema for courses and jobs:
## courses:
|university| department| faculty| code| title| course_type| description| skills|

### For NUS:
    keep:
        "department"
        "description"
        "faculty"
        "moduleCode"
        "title"
        
    ignore:
        "gradingBasisDescription"
        "moduleCredit"
        "semesterData"
        "workload"


### For NTU:
    keep:
        "code" 
        "title" 
        "description" 
        "departmentMaintaining"

    ignore:
        "academicUnits"
        "examSchedule" 
        "gradeType" 
        "prerequisites" 
        "notAvailableToProgramme" (eg not available to EEE)
        "url" 
        "details" 


### For SUTD:
    keep:
        "code"
        "title" 
        "course_type" (core, electives, etc)
        "description" 
        "affiliations" (department)

    ignore:
        "terms" 
        "credits" 
        "url" 
        "description_found" (T/F)


extract skills from the descriptions



## jobs:
### keep: 
        "title"
        "description"
        "minimumYearsExperience"
        "shiftPattern"
        "skills" (keep everything in the skills section)
            "uuid": "11ff9f68afb6b8b5b8eda218d7c83a65",
            "confidence": null,
            "isKeySkill": null
        "otherRequirements"
        "ssocCode" (job industry/sector)
        "occupationId"
        "ssocVersion"
        "workingHours"
        "numberOfVacancies"
        "categories"
            "id"
            "category"
        "employmentTypes" 
            "id" 
            "employmentType" 
        "positionLevels"
            "id": 11,
            "position" 
        "ssecEqa" 
            (Singapore Standard Educational Classification (SSEC) Educational Qualification Assessment code, indicating the education level. 
            eg "69" typically corresponds to a Bachelor's Degree with Honours.)
        "ssecFos" 
            (Field of Study code, 
            eg "0212" corresponds to "Music and Performing Arts")
        "ssicCode" 
            (It classifies the type of industry/business sector the company operates in.)
            (SSOC = the job classification (Laboratory Technician, Finance Manager, etc.)
            (SSIC = the company/industry classification (Manufacturing, Education, etc.))
        "ssicCode2020" 
         "salary": {
                    "maximum": 3500,
                    "minimum": 2500,
                    "type": {
                    "id": 4,
                    "salaryType": "Monthly"
                    }
        "jobTitles"

### keep but tbc: (if all NA/nulls -> drop)
        "schemes" 
        "flexibleWorkArrangements"

### ignore:
        "uuid" (post id? user id?)
        "sourceCode" 
        "psdUrl"
        "status"
        "postedCompany" 
            "uen" 
                ("uen" stands for Unique Entity Number, a unique identifier assigned to all business entities (companies, organizations, partnerships) registered in Singapore. In the example, "52875677W" is the UEN for SYMPHONY MUSIC SCHOOL. It's used to uniquely identify and track companies in Singapore's business registry.)
            "description" (company description)
            "name" (company name)
        "employeeCount" 
        "companyUrl"
        "lastSyncDate": "2026-01-30T02:57:44.000Z",
        "_links"
        "logoFileName" 
        "logoUploadPath" 
        "responsiveEmployer": {
      "isResponsive": false
    }



# NUS

## Load Data

In [51]:
# import necessary libraries
import json5
import pandas as pd
import numpy as np
from pathlib import Path

In [52]:
# load output NUSModsInfo.json file
nus_file_path = Path("../../data/NUSModsInfo.json")

with open(nus_file_path, "r", encoding="utf-8") as f:
    nus_data = json5.load(f)

In [53]:
# Extract relevant fields
nus_fields = ["department", "description", "faculty", "moduleCode", "title"]

nus_filtered_data = [
    {key: item.get(key) for key in nus_fields}
    for item in nus_data
]

In [54]:
# Create DataFrame
nus_df = pd.DataFrame(nus_filtered_data)

# Preview
nus_df.head()

,department,description,faculty,moduleCode,title
0,NUS Medicine Dean's Office,Leadership is fundamental to the success of in...,Yong Loo Lin Sch of Medicine,ABM5001,Leadership in Biomedicine
1,NUS Medicine Dean's Office,This course serves as a concept-based introduc...,Yong Loo Lin Sch of Medicine,ABM5002,Advanced Biostatistics for Research
2,NUS Medicine Dean's Office,This course will furnish students with a thoro...,Yong Loo Lin Sch of Medicine,ABM5003,Biomedical Innovation & Enterprise
3,NUS Medicine Dean's Office,This course encompasses research projects rele...,Yong Loo Lin Sch of Medicine,ABM5004,Capstone Project
4,NUS Medicine Dean's Office,Advanced immunological applications play impor...,Yong Loo Lin Sch of Medicine,ABM5101,Applied Immunology


## Data Cleaning

### Data Exploration

In [55]:
# check for missing values
print(nus_df.isnull().sum())

department     103
description      0
faculty          0
moduleCode       0
title            0
dtype: int64


In [56]:
nus_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20517 entries, 0 to 20516
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   department   20414 non-null  object
 1   description  20517 non-null  object
 2   faculty      20517 non-null  object
 3   moduleCode   20517 non-null  object
 4   title        20517 non-null  object
dtypes: object(5)
memory usage: 801.6+ KB


In [57]:
# unique values in department and faculty
print("Unique departments:", nus_df["department"].nunique())
print("Unique faculties:", nus_df["faculty"].nunique()) 

Unique departments: 109
Unique faculties: 25


In [58]:
# get unique faculties 
unique_faculties = nus_df["faculty"].unique()
print("Unique faculties:", unique_faculties)

Unique faculties: ['Yong Loo Lin Sch of Medicine' 'College of Design and Engineering'
 'NUS Business School' 'Arts and Social Science' 'NUS' 'Computing' 'Law'
 'Cont and Lifelong Education' 'Science' 'Non-Faculty-based Departments'
 'Dentistry' 'YST Conservatory of Music' 'Multi Disciplinary Programme'
 'NUS-ISS' 'Residential College' 'Temasek Defence Sys. Institute'
 'Risk Management Institute' 'NUS College' 'SSH School of Public Health'
 'Duke-NUS Medical School' 'NUS Graduate School' 'Logistics Inst-Asia Pac'
 'Mechanobiology Institute (MBI)' 'LKY School of Public Policy'
 'Yale-NUS College']


In [59]:
# faculty level data distribution
faculty_counts = nus_df["faculty"].value_counts()
print(faculty_counts)

faculty
Arts and Social Science              5679
Law                                  2878
College of Design and Engineering    2823
Science                              1695
NUS Business School                  1642
Yale-NUS College                     1485
Computing                             541
Yong Loo Lin Sch of Medicine          517
Cont and Lifelong Education           425
Duke-NUS Medical School               416
LKY School of Public Policy           377
YST Conservatory of Music             373
Non-Faculty-based Departments         366
NUS College                           337
Residential College                   261
SSH School of Public Health           208
NUS                                   154
NUS-ISS                                82
NUS Graduate School                    65
Dentistry                              59
Temasek Defence Sys. Institute         51
Risk Management Institute              48
Multi Disciplinary Programme           15
Mechanobiology Institute (

### Filter for undergraduate courses only

In [60]:
# Keep only undergraduate level courses: codes where the first digit after letters is 0-4
nus_df = nus_df[nus_df['moduleCode'].str.match(r'^[A-Z]+[0-4]')]

nus_df

,department,description,faculty,moduleCode,title
27,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701,Accounting for Decision Makers
28,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701A,Accounting for Decision Makers
29,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701B,Accounting for Decision Makers
30,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701C,Accounting for Decision Makers
31,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701D,Accounting for Decision Makers
...,...,...,...,...,...
20512,Biological Sciences,In addition to having an academic science foun...,Science,ZB3312,Enhanced Undergraduate Professional Internship...
20513,Biological Sciences,In addition to having an academic science foun...,Science,ZB3313,Undergraduate Professional Internship Programm...
20514,Biological Sciences,This is a seminar-style course based on the li...,Science,ZB4171,Advanced Topics in Bioinformatics
20515,Biological Sciences,Not Available,Science,ZB4199,Honours Project in Computational Biology


In [61]:
# faculty level data distribution
faculty_counts = nus_df["faculty"].value_counts()
print(faculty_counts)

faculty
Arts and Social Science              4631
Yale-NUS College                     1485
College of Design and Engineering    1374
Science                              1172
NUS Business School                   796
Law                                   783
Cont and Lifelong Education           403
Computing                             350
YST Conservatory of Music             337
NUS College                           337
Non-Faculty-based Departments         333
Residential College                   261
Yong Loo Lin Sch of Medicine          174
NUS                                   122
SSH School of Public Health            61
Dentistry                              50
NUS-ISS                                19
Multi Disciplinary Programme           15
Duke-NUS Medical School                 3
Temasek Defence Sys. Institute          2
Name: count, dtype: int64


In [62]:
# filter out post graduate level courses: faculty is for post graduate level courses, which are not relevant to our analysis
target_faculties = [
    "Temasek Defence Sys. Institute",
     "Cont and Lifelong Education",
     "Duke-NUS Medical School"
]

filtered_nus_df = nus_df[~nus_df["faculty"].isin(target_faculties)]

### Handling NA Values

In [63]:
# count data with null description
null_description_count = filtered_nus_df["description"].isnull().sum()
print("Number of courses with null description:", null_description_count)

Number of courses with null description: 0


In [64]:
# Distribution of description data
description_counts = filtered_nus_df["description"].value_counts()
print(description_counts)

description
Not Available                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           3664
UTOP aims to train undergraduates to acquire an

some short descriptions such as: NIL, Exchange Course - YUS (1 unit), Not Applicable 
are not meaningful for analysis. we would remove those data

In [65]:
# Keep only rows where description has 10 or more words
filtered_nus_df = filtered_nus_df[filtered_nus_df['description'].str.split().str.len() >= 10]

In [66]:
filtered_nus_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8499 entries, 27 to 20516
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   department   8499 non-null   object
 1   description  8499 non-null   object
 2   faculty      8499 non-null   object
 3   moduleCode   8499 non-null   object
 4   title        8499 non-null   object
dtypes: object(5)
memory usage: 398.4+ KB


there is no more NA data in the dataset

## Data Transformation

In [67]:
filtered_nus_df["University"] = "NUS"

In [68]:
filtered_nus_df.head()

,department,description,faculty,moduleCode,title,University
27,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701,Accounting for Decision Makers,NUS
28,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701A,Accounting for Decision Makers,NUS
29,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701B,Accounting for Decision Makers,NUS
30,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701C,Accounting for Decision Makers,NUS
31,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701D,Accounting for Decision Makers,NUS


# NTU

## Load Data

In [69]:
# load output ntuCourseInfo.json file
ntu_file_path = Path("../../data/ntuCourseInfo.json")

with open(ntu_file_path, "r", encoding="utf-8") as f:
    ntu_data = json5.load(f)

In [70]:
# Extract relevant fields
ntu_fields =  ["code", "title", "description", "departmentMaintaining"]

ntu_filtered_data = [
    {key: item.get(key) for key in ntu_fields}
    for item in ntu_data
]

In [71]:
# Create DataFrame
ntu_df = pd.DataFrame(ntu_filtered_data)

# Preview
ntu_df.head()

,code,title,description,departmentMaintaining
0,AAA08B,AAA08B FASHION & DESIGN: WEARABLE ART AS A SEC...,Develop process skills in expression and visua...,NIE
1,AAA08C,AAA08C EXPRESSIVE DRAWING: DEVELOPING PERSONAL...,Students will learn a brief history of express...,NIE
2,AAA08D,AAA08D ABSTRACT PAINTING: WHY IT'S HERE & HOW ...,Students will learn a brief history of abstrac...,NIE
3,AAA18D,AAA18D LIFE DRAWING,This course introduces drawing of the nude hum...,NIE
4,AAA18E,AAA18E DRAWING,This course introduces drawing at a basic leve...,NIE


## Data Cleaning

### Data Exploration

In [72]:
# check for missing values
print(ntu_df.isnull().sum())

code                      0
title                     0
description               0
departmentMaintaining    14
dtype: int64


In [73]:
ntu_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2117 entries, 0 to 2116
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   code                   2117 non-null   object
 1   title                  2117 non-null   object
 2   description            2117 non-null   object
 3   departmentMaintaining  2103 non-null   object
dtypes: object(4)
memory usage: 66.3+ KB


In [74]:
# unique values in department
print("Unique departments:", ntu_df["departmentMaintaining"].nunique())

Unique departments: 47


In [75]:
# faculty level data distribution
ntu_dept_counts = ntu_df["departmentMaintaining"].value_counts()
print(ntu_dept_counts)

departmentMaintaining
BUS          144
ADM          144
CSC(CE)      129
NIE          119
SOH          111
EEE           96
CS            96
ELH(SOH)      74
MATH(SPS)     71
CHIN(SOH)     67
ME            64
BS            62
LMS(SOH)      57
PSY(SSS)      56
EESS(ASE)     54
HIST(SOH)     53
MAT           53
ECON(SSS)     51
PHY(SPS)      49
CBE           49
PPGA(SSS)     49
PHIL(SOH)     48
SOC(SSS)      41
CEE           39
CHEM(CBE)     37
CE            36
ACC           36
BIE(CBE)      28
REP           25
SSM(NIE)      22
MS(CEE)       21
ENE(CEE)      20
AERO(ME)      20
CMED(BS)      15
NTC           14
BMS(BS)       13
SSS            9
ICC            8
SPS            6
IEM(EEE)       4
BSB(BS)        4
ROBO(ME)       3
BACF(CE)       2
CAO            1
BSB            1
ACDA(CE)       1
BMS            1
Name: count, dtype: int64


### Filter for undergraduate courses only

In [76]:
# Keep only undergraduate level courses: codes where the first digit after letters is 0-4
ntu_df = ntu_df[ntu_df['code'].str.match(r'^[A-Z]+[0-4]')]

### Add department name

In [77]:
# import openpyxl if you can't load
# import openpyxl

In [78]:
ntu_dept_file_path = Path("../../data/ntu_dept_mapping.xlsx")

ntu_dept_df = pd.read_excel(ntu_dept_file_path)

ntu_dept_df.head()


,short,department
0,ACC,Nanyang Business School (Accountancy)
1,ACDA(CE),Programme under Civil Engineering
2,ADM,"School of Art, Design and Media ..."
3,AERO(ME),Aerospace Engineering (MAE)
4,BACF(CE),Bachelor of Applied Computing in Finance (CE)


In [79]:
# add department mapping to ntu_df
ntu_df = ntu_df.merge(
    ntu_dept_df[["short", "department"]],
    left_on="departmentMaintaining",
    right_on="short",
    how="left"
).drop(columns=["short"])

In [80]:
ntu_df.rename(columns={"departmentMaintaining": "dept_code"}, inplace=True)

### Handling NA Values

In [81]:
# count data with null description
ntu_null_description_count = ntu_df["description"].isna().sum()
print("Number of courses with null description:", ntu_null_description_count)

Number of courses with null description: 0


In [82]:
# Distribution of description data
ntu_description_counts = ntu_df["description"].value_counts()
print(ntu_description_counts)

description
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [83]:
# Keep only rows where description has 10 or more words
filtered_ntu_df = ntu_df[ntu_df['description'].str.split().str.len() >= 10]

In [84]:
filtered_ntu_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1818 entries, 0 to 1851
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   code         1818 non-null   object
 1   title        1818 non-null   object
 2   description  1818 non-null   object
 3   dept_code    1818 non-null   object
 4   department   1818 non-null   object
dtypes: object(5)
memory usage: 85.2+ KB


## Data Transformation

In [85]:
filtered_ntu_df["University"] = "NTU"

/var/folders/gn/g9t9ks6j5qq32pcsbtb27jxr0000gn/T/ipykernel_50087/858357989.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_ntu_df["University"] = "NTU"


In [86]:
filtered_ntu_df.head()

,code,title,description,dept_code,department,University
0,AAA08B,AAA08B FASHION & DESIGN: WEARABLE ART AS A SEC...,Develop process skills in expression and visua...,NIE,National Institute of Education,NTU
1,AAA08C,AAA08C EXPRESSIVE DRAWING: DEVELOPING PERSONAL...,Students will learn a brief history of express...,NIE,National Institute of Education,NTU
2,AAA08D,AAA08D ABSTRACT PAINTING: WHY IT'S HERE & HOW ...,Students will learn a brief history of abstrac...,NIE,National Institute of Education,NTU
3,AAA18D,AAA18D LIFE DRAWING,This course introduces drawing of the nude hum...,NIE,National Institute of Education,NTU
4,AAA18E,AAA18E DRAWING,This course introduces drawing at a basic leve...,NIE,National Institute of Education,NTU


# SUTD

## Load Data

In [87]:
# load output from sutdCourseDescriptions.json file
sutd_file_path = Path("../../data/sutdCourseDescriptions.json")

with open(sutd_file_path, "r", encoding="utf-8") as f:
    sutd_data = json5.load(f)

In [88]:
# Extract relevant fields
sutd_fields =  ["code", "title", "course_type", "description", "affiliations"]

sutd_filtered_data= [
    {key: item.get(key) for key in sutd_fields}
    for item in sutd_data
]

In [89]:
# Create DataFrame
sutd_df = pd.DataFrame(sutd_filtered_data)

sutd_df.head()

,code,title,course_type,description,affiliations
0,01.018,Design Thinking Project I,Freshmore core,"Design Thinking Projects I, II, and III provid...",[SMT]
1,01.019,Design Thinking Project II,Freshmore core,"Design Thinking Projects I, II, and III provid...","[EPD, SMT]"
2,01.020,Design Thinking Project III,Freshmore core,"Design Thinking Projects I, II, and III provid...",[SMT]
3,01.101,Technologies for Sustainable Global Health,Elective / Technical Elective,This course provides a multi-disciplinary appr...,"[EPD, SMT]"
4,01.102,Energy Systems and Management,Elective / Technical Elective,"Broadly, this course aims to acquaint students...",[ESD]


## Data Cleaning

### Data Exploration

In [90]:
# check for missing values
print(sutd_df.isnull().sum())

code            0
title           0
course_type     0
description     5
affiliations    0
dtype: int64


### Add department name

In [91]:
AFFILIATION_NAMES = {
    "ASD": "Architecture and Sustainable Design",
    "DAI": "Design and Artificial Intelligence",
    "EPD": "Engineering Product Development",
    "ESD": "Engineering Systems and Design",
    "HASS": "Humanities, Arts and Social Sciences",
    "ISTD": "Information Systems Technology and Design",
    "SMT": "Science, Mathematics and Technology",
}

sutd_df["department"] = sutd_df["affiliations"].apply(
    lambda codes: [AFFILIATION_NAMES[c] for c in (codes or []) if c in AFFILIATION_NAMES]
)

### Handling NA Values

In [92]:
# count data with null description
sutd_null_description_count = sutd_df["description"].isna().sum()
print("Number of courses with null description:", sutd_null_description_count)

Number of courses with null description: 5


In [93]:
# Distribution of description data
sutd_description_counts = sutd_df["description"].value_counts()
print(sutd_description_counts)

description
Design Thinking Projects I, II, and III provide the Freshmore students the opportunity to learn and practice fundamental design thinking principles. Students are introduced to design thinking through a series of design-centric, interdisciplinary, multidisciplinary, hands-on projects and seminars, guided by a yearly general theme to ensure integrated pedagogy and progressive learning. Moreover, the projects and their solutions are expected to impact areas of growth at SUTD, such as healthcare, cities, aviation, and data science. 01.018 Design Thinking Project I in term 1 is a guided project, 01.019 Design Thinking Projects II in term 2 is integrated with 3.007 Design Thinking and Innovation course, while 01.020 Design Thinking Project III in term 3 allows the students with more scope for self-guidance. Ultimately, the courses are designed to equip the students with a series of design thinking, technical, contextual, organizational, leadership, scientific and technological sk

In [94]:
# remove courses with description less than 10 words and NA descriptions
filtered_sutd_df = sutd_df[sutd_df['description'].str.split().str.len() >= 10]

## Data Transformation

In [95]:
filtered_sutd_df["University"] = "SUTD"

/var/folders/gn/g9t9ks6j5qq32pcsbtb27jxr0000gn/T/ipykernel_50087/1833491157.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_sutd_df["University"] = "SUTD"


In [96]:
filtered_sutd_df.head()

,code,title,course_type,description,affiliations,department,University
0,01.018,Design Thinking Project I,Freshmore core,"Design Thinking Projects I, II, and III provid...",[SMT],"[Science, Mathematics and Technology]",SUTD
1,01.019,Design Thinking Project II,Freshmore core,"Design Thinking Projects I, II, and III provid...","[EPD, SMT]","[Engineering Product Development, Science, Mat...",SUTD
2,01.020,Design Thinking Project III,Freshmore core,"Design Thinking Projects I, II, and III provid...",[SMT],"[Science, Mathematics and Technology]",SUTD
3,01.101,Technologies for Sustainable Global Health,Elective / Technical Elective,This course provides a multi-disciplinary appr...,"[EPD, SMT]","[Engineering Product Development, Science, Mat...",SUTD
4,01.102,Energy Systems and Management,Elective / Technical Elective,"Broadly, this course aims to acquaint students...",[ESD],[Engineering Systems and Design],SUTD


# All Uni Courses Dataframe


|university| department| faculty| code| title| course_type| description| skills|

In [97]:
# Standardize column names for consistency
# NUS: rename 'moduleCode' to 'code'
filtered_nus_df = filtered_nus_df.rename(columns={'moduleCode': 'code'})

# NTU: already has 'code', but 'dept_code' and 'department' are separate
# Keep 'department' as the full name, drop 'dept_code' if not needed

# SUTD: already has 'code', 'department' is a list, but we'll keep it

# Create unified columns: ['university', 'department', 'faculty', 'code', 'title', 'course_type', 'description']

# For NUS
nus_unified = filtered_nus_df[['University', 'department', 'faculty', 'code', 'title', 'description']].copy()
nus_unified['course_type'] = None  # NUS doesn't have course_type

# For NTU
ntu_unified = filtered_ntu_df[['University', 'department', 'code', 'title', 'description']].copy()
ntu_unified['faculty'] = None  # NTU doesn't have faculty
ntu_unified['course_type'] = None

# For SUTD
sutd_unified = filtered_sutd_df[['University', 'department', 'code', 'title', 'course_type', 'description']].copy()
sutd_unified['faculty'] = None  # SUTD doesn't have faculty

# Concatenate all courses
all_courses_df = pd.concat([nus_unified, ntu_unified, sutd_unified], ignore_index=True)

# Rename columns to match target schema
all_courses_df = all_courses_df.rename(columns={'University': 'university'})

print(f"Consolidated courses shape: {all_courses_df.shape}")
all_courses_df.head()

Consolidated courses shape: (10516, 7)


,university,department,faculty,code,title,description,course_type
0,NUS,Accounting,NUS Business School,ACC1701,Accounting for Decision Makers,The course provides an introduction to account...,None
1,NUS,Accounting,NUS Business School,ACC1701A,Accounting for Decision Makers,The course provides an introduction to account...,None
2,NUS,Accounting,NUS Business School,ACC1701B,Accounting for Decision Makers,The course provides an introduction to account...,None
3,NUS,Accounting,NUS Business School,ACC1701C,Accounting for Decision Makers,The course provides an introduction to account...,None
4,NUS,Accounting,NUS Business School,ACC1701D,Accounting for Decision Makers,The course provides an introduction to account...,None


# Text Preprocessing  
- Cleans raw text in the description columns for both courses and jobs.
- Removes HTML tags (e.g., <p> tags from job descriptions).
- Strips extra whitespace and lowercases everything.
- Creates new columns: description_clean for both all_courses_df and df_jobs.

In [98]:
import re
from bs4 import BeautifulSoup

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = BeautifulSoup(text, "html.parser").get_text()
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    return text

# Apply to courses
all_courses_df['description_clean'] = all_courses_df['description'].apply(clean_text)

print("Text preprocessing completed for courses.")

Text preprocessing completed for courses.


# Save Cleaned Data

In [99]:
print("\n=== Courses Validation ===")

print(f"Shape: {all_courses_df.shape}")
print(f"Null counts:\n{all_courses_df.isnull().sum()}")
print(f"\nData types:\n{all_courses_df.dtypes}")
print(f"\nUnique universities: {all_courses_df['university'].unique()}")

# Data quality check
courses_no_description = (
    all_courses_df['description'].isna() |
    (all_courses_df['description'] == '')
).sum()

print(f"\nCourses without description: {courses_no_description}")

if courses_no_description > 0:
    print("Removing courses without descriptions...")
    all_courses_df = all_courses_df[
        all_courses_df['description'].notna() &
        (all_courses_df['description'] != '')
    ]
    print(f"Courses after cleanup: {all_courses_df.shape[0]}")

print("\n✓ Courses validation complete")


=== Courses Validation ===
Shape: (10516, 8)
Null counts:
university               0
department               0
faculty               2017
code                     0
title                    0
description              0
course_type          10317
description_clean        0
dtype: int64

Data types:
university           object
department           object
faculty              object
code                 object
title                object
description          object
course_type          object
description_clean    object
dtype: object

Unique universities: ['NUS' 'NTU' 'SUTD']

Courses without description: 0

✓ Courses validation complete


In [100]:
from pathlib import Path

output_dir = Path("../../data/cleaned")
output_dir.mkdir(exist_ok=True)

all_courses_df.to_pickle(output_dir / "courses_cleaned.pkl")

print(f"Saved courses data to {output_dir}")
print(f"Courses: {all_courses_df.shape[0]} records")

Saved courses data to ../../data/cleaned
Courses: 10516 records
